In [1]:
# ## Morphology Operations Benchmark Notebook
# 
# This notebook will run a set of morphology operations on a user-provided image and kernel,
# measuring execution time and accuracy against `scikit-image` functions where available.
# Results will be saved to a CSV file.

import numpy as np
import pandas as pd
import timeit
from skimage.morphology import binary_erosion, binary_dilation, binary_closing, binary_opening

# workaround to allow importing harpia python module
import sys

sys.path.append("../../")
from harpia.morphology.operations_binary import (
    erosion_binary,
    dilation_binary,
    closing_binary,
    opening_binary,
    smooth_binary,
    geodesic_erosion_binary,
    geodesic_dilation_binary,
    reconstruction_binary,
    fill_holes
)

In [9]:
def load_image(path, xsize, ysize, zsize, dtype, dtype_out):
    img = np.fromfile(path, dtype=dtype)
    img = img.reshape((zsize, ysize, xsize))
    img = img.astype(dtype = dtype_out)
    return img
    
def custum_kernel3D():
    kernel_2d = np.array([[1, 1, 1], [1, 1, 1], [1, 1, 1]], dtype=np.int32)
    # Stack the 2D kernel to form a 3D kernel (3 layers)
    kernel_3d = np.stack([kernel_2d, kernel_2d, kernel_2d])
    return kernel_3d

In [8]:
# Specify the image and kernel
def load_image_and_kernel():
    xsize = 2048
    ysize = 2048
    zsize = 1964
    path = "../../../../../../../../beamlines/mogno/proposals/20180217/data/Soil_Experiment/testes_segmentacao/PBV29_Talita/tomo_NLM_masked_2048x2048x1964_16bit.raw"
    image = np.fromfile(path, dtype='int16')
    image = image.reshape((zsize, ysize, xsize))
    image = image.astype(dtype = 'int')
    kernel = np.ones((3, 3, 3), dtype='int')  # Define a sample kernel or load one
    return image, kernel

# Helper function to time and compare results
def time_and_compare(module_func, skimage_func, image, kernel, csv_data):
    # Timing the module function
    start = timeit.default_timer()
    module_output = module_func(image, kernel)
    module_time = timeit.default_timer() - start

    # Timing the scikit-image function
    start = timeit.default_timer()
    skimage_output = skimage_func(image, kernel)
    skimage_time = timeit.default_timer() - start

    # Check if outputs match
    accuracy = np.array_equal(module_output, skimage_output)
    
    # Add to CSV data
    csv_data.append({
        'Operation': module_func.__name__,
        'Module Time (s)': module_time,
        'Scikit-Image Time (s)': skimage_time,
        'Accuracy': accuracy
    })
    return module_output

# For additional functions without scikit-image equivalent, run only the module function timing
def time_module_only(module_func, image, kernel, csv_data):
    start = timeit.default_timer()
    module_output = module_func(image, kernel)
    module_time = timeit.default_timer() - start
    
    # Add to CSV data
    csv_data.append({
        'Operation': module_func.__name__,
        'Module Time (s)': module_time,
        'Scikit-Image Time (s)': 'N/A',
        'Accuracy': 'N/A'
    })

In [19]:
# original img
xsize = 2048
ysize = 2048
zsize_original = 1964
zsize = 100
path = "../../../../../../../../beamlines/mogno/proposals/20180217/data/Soil_Experiment/testes_segmentacao/PBV29_Talita/Matriz_poros/total/poros_matriz_2048x2048x1964_8bit.raw"

image = load_image(path, xsize, ysize, zsize_original,'int8', 'int32')
image = image[:zsize,:,:] #reduce size

kernel = custum_kernel3D()

In [22]:
# Run and log each test
csv_data = []

# Erosion Binary Test
time_and_compare(erosion_binary, binary_erosion, image, kernel, csv_data)

# Dilation Binary Test
time_and_compare(dilation_binary, binary_dilation, image, kernel, csv_data)

# Closing Binary Test
time_and_compare(closing_binary, binary_closing, image, kernel, csv_data)

# Opening Binary Test
time_and_compare(opening_binary, binary_opening, image, kernel, csv_data)

# Smooth binary, geodesic operations, and others without direct skimage equivalents
time_module_only(smooth_binary, image, kernel, csv_data)

# Add similar calls for other functions: geodesic_erosion_binary, geodesic_dilation_binary, reconstruction_binary, fill_holes

# Save results to CSV
results_df = pd.DataFrame(csv_data)
results_df.to_csv("morphology_benchmark_results.csv", index=False)

print("Benchmark completed and saved to morphology_benchmark_results.csv")

Benchmark completed and saved to morphology_benchmark_results.csv


In [23]:
results_df

,Operation,Module Time (s),Scikit-Image Time (s),Accuracy
0,erosion_binary,0.490576,3.053128,False
1,dilation_binary,0.496649,13.273286,False
2,closing_binary,0.522772,15.786618,False
3,opening_binary,0.524637,15.207348,False
4,smooth_binary,0.589140,N/A,N/A
